In [ ]:
import torch
import torch_geometric
import rdkit
import networkx

print("✅ PyTorch version:", torch.__version__)
print("✅ CUDA available:", torch.cuda.is_available())
print("✅ PyG version:", torch_geometric.__version__)

✅ PyTorch version: 2.6.0+cu126
✅ CUDA available: False
✅ PyG version: 2.6.1


In [ ]:
# Import required libraries
import torch
import torch.nn as nn
import torch.optim as optim
import networkx as nx  # For molecular graphs
from rdkit import Chem  # For molecular validity checks
from torch_geometric.nn import GCNConv  # Graph Neural Networks (GNNs)
from torch_geometric.data import Data  # Graph data format

In [ ]:
# -------------------
# Define Generator (Graph-Based)
# -------------------
class GraphGenerator(nn.Module):
    def __init__(self, latent_dim, node_features, edge_features):
        super(GraphGenerator, self).__init__()
        self.fc = nn.Linear(latent_dim, node_features * edge_features)
        self.gnn = GCNConv(node_features, edge_features)
    
    def forward(self, z):
        # Map latent space to graph structure
        node_embeddings = torch.relu(self.fc(z))
        edge_index = self.create_edges(node_embeddings)
        return node_embeddings, edge_index

    def create_edges(self, node_embeddings):
        # Generate edges probabilistically (for valid molecules)
        num_nodes = node_embeddings.shape[0]
        edges = [(i, j) for i in range(num_nodes) for j in range(num_nodes) if i != j]
        edge_index = torch.tensor(edges, dtype=torch.long).t().contiguous()
        return edge_index

In [ ]:
# -------------------
# Define Discriminator (WGAN-based)
# -------------------
class GraphDiscriminator(nn.Module):
    def __init__(self, node_features):
        super(GraphDiscriminator, self).__init__()
        self.conv1 = GCNConv(node_features, 64)
        self.conv2 = GCNConv(64, 32)
        self.fc = nn.Linear(32, 1)

    def forward(self, node_embeddings, edge_index):
        x = torch.relu(self.conv1(node_embeddings, edge_index))
        x = torch.relu(self.conv2(x, edge_index))
        return torch.sigmoid(self.fc(x.mean(dim=0)))  # Single validity score

In [ ]:
# -------------------
# Define Reinforcement Learning Component
# -------------------
class RewardFunction:
    def __init__(self):
        pass

    def compute_reward(self, molecule):
        # Evaluate fuel properties (e.g., energy density, emissions)
        reward = 0
        if self.is_valid(molecule):
            reward += 10  # Structural validity reward
        if self.optimised_fuel_properties(molecule):
            reward += 20  # Fuel-specific property reward
        return reward

    def is_valid(self, molecule):
        # Validate molecule using RDKit
        mol = Chem.MolFromSmiles(molecule)
        return mol is not None

    def optimised_fuel_properties(self, molecule):
        # Placeholder for domain-specific fuel evaluation
        return True  # Assume valid for now

In [ ]:
# -------------------
# Define Training Loop
# -------------------
def train_GAN(num_epochs, generator, discriminator, reward_fn, latent_dim):
    optimizer_G = optim.Adam(generator.parameters(), lr=0.0002)
    optimizer_D = optim.Adam(discriminator.parameters(), lr=0.0002)
    
    for epoch in range(num_epochs):
        # Generate latent vector
        z = torch.randn((batch_size, latent_dim))

        # Generate molecular graph
        generated_nodes, generated_edges = generator(z)
        
        # Compute reward using reinforcement learning
        reward = reward_fn.compute_reward(generated_nodes)

        # Train Discriminator (WGAN-GP loss)
        real_score = discriminator(real_nodes, real_edges)  # Real molecules
        fake_score = discriminator(generated_nodes, generated_edges)  # Fake molecules
        loss_D = -real_score + fake_score  # Wasserstein loss
        optimizer_D.zero_grad()
        loss_D.backward()
        optimizer_D.step()

        # Train Generator (Reward Maximisation)
        loss_G = -fake_score + reward  # Encourage valid fuel molecules
        optimizer_G.zero_grad()
        loss_G.backward()
        optimizer_G.step()

        # Logging
        if epoch % 10 == 0:
            print(f"Epoch [{epoch}/{num_epochs}], Loss_D: {loss_D.item()}, Loss_G: {loss_G.item()}")

In [ ]:
# -------------------
# Main Execution
# -------------------
if __name__ == "__main__":
    latent_dim = 128
    node_features = 10  # Placeholder, adjust for dataset
    edge_features = 5   # Placeholder, adjust for dataset
    num_epochs = 500
    batch_size = 32

    generator = GraphGenerator(latent_dim, node_features, edge_features)
    discriminator = GraphDiscriminator(node_features)
    reward_fn = RewardFunction()

    train_GAN(num_epochs, generator, discriminator, reward_fn, latent_dim)

TypeError: No registered converter was able to produce a C++ rvalue of type std::basic_string<wchar_t, std::char_traits<wchar_t>, std::allocator<wchar_t> > from this Python object of type Tensor